# Laboratorio inicial — primer contacto con los datos

Ejecute este notebook desde la raíz del repositorio o desde `notebooks/`: la
primera celda resuelve la ruta sola.

**Si está en una simulación cronometrada, no abra `solutions/`.** Este notebook es
el punto de partida, no la solución.

Los datos son sintéticos, generados con semilla fija. No describen ningún sistema
real.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# La raíz del repositorio, venga de donde venga el kernel.
root = Path.cwd()
if not (root / "data").exists():
    root = root.parent
sys.path.insert(0, str(root))

CASO = 1  # cambie a 2 o 3 para los casos mayores
carpeta = root / "data" / f"case_{CASO}"
assert carpeta.exists(), "Ejecute primero: python scripts/generate_data.py"
print(root)

## 1. Leer sin interpretar

Primero se mira el archivo como está. Convertir tipos antes de medir oculta
precisamente lo que hay que reportar.

In [ ]:
viajes = pd.read_csv(carpeta / "viajes.csv")
recorridos = pd.read_csv(carpeta / "recorridos.csv")
operadores = pd.read_csv(carpeta / "operadores.csv")
programacion = pd.read_csv(carpeta / "programacion.csv")

print({
    "viajes": viajes.shape,
    "recorridos": recorridos.shape,
    "operadores": operadores.shape,
    "programacion": programacion.shape,
})
viajes.head()

## 2. Perfilar

`lab.core.profile` entrega, por columna, el tipo, los nulos, los valores únicos y
los duplicados exactos de la tabla. Es el primer comando de cualquier análisis.

In [ ]:
from lab.core import profile

profile(viajes)

## 3. Validar contra un contrato

El contrato declara lo que la tabla debe cumplir. La validación **no elimina filas**:
informa cada incumplimiento con su regla, su columna, su recuento y su severidad. La
decisión de excluir es del análisis, y hay que declararla.

In [ ]:
from lab.contracts import (
    Column,
    ForeignKey,
    TableContract,
    findings_frame,
    validate_dataset,
)

contrato_viajes = TableContract(
    name="viajes",
    columns=(
        Column("viaje_id", "texto", nullable=False, unique=True),
        Column("recorrido_id", "texto", nullable=False),
        Column("inicio", "fecha_hora"),
        Column("fin", "fecha_hora"),
        Column("pasajeros", "entero", minimum=0, unit="personas"),
        Column("estado", "texto", allowed=("REALIZADO", "CANCELADO")),
    ),
    primary_key=("viaje_id",),
    foreign_keys=(ForeignKey("recorrido_id", "recorridos", "recorrido_id"),),
)
contrato_recorridos = TableContract(
    name="recorridos",
    columns=(
        Column("recorrido_id", "texto", nullable=False, unique=True),
        Column("operador_id", "texto", nullable=False),
        Column("meta_min", "decimal", minimum=0, unit="min"),
    ),
    primary_key=("recorrido_id",),
)

informes = validate_dataset(
    [contrato_viajes, contrato_recorridos],
    {"viajes": viajes, "recorridos": recorridos},
)
findings_frame(informes.values())

## 4. Calidad de las fechas

`date_quality` separa el valor **vacío** del valor **no interpretable**. Son dos
defectos distintos, y en los viajes cancelados el `fin` está vacío por diseño.

In [ ]:
from lab.dates import date_quality

print("inicio")
print(date_quality(viajes["inicio"]).to_string(index=False))
print("\nfin")
print(date_quality(viajes["fin"]).to_string(index=False))

## 5. Un indicador con su cobertura

Una media sin cobertura no se puede interpretar: no se sabe sobre qué parte de la
población se calculó.

In [ ]:
from lab.metrics import describe_numeric

describe_numeric(viajes["pasajeros"], unit="personas")

## Ejercicio

Con lo anterior como punto de partida:

1. Elimine los duplicados exactos **contando** cuántos elimina.
2. Convierta tipos con `errors="coerce"` e informe cuántos valores se perdieron.
3. Cruce con `recorridos` y `operadores` usando `validate="many_to_one"`, y cuente
   las claves desconocidas.
4. Cree columnas de bandera (`realizado`, `pax_valido`, `dur_valida`) en lugar de ir
   filtrando y reasignando `viajes`.
5. Calcule demanda por operador y por hora, y duración por recorrido, cada una sobre
   su propio conjunto válido.
6. Exporte con `lab.excel.write_workbook`, separando entradas, cálculos y resultados.
7. Compruebe que la suma por hora coincide con el total general.

Si se traba, la clase
`docs/learning/20-programacion/01-pandas-para-gestion-de-datos.md` tiene el recorrido
completo comentado.

In [ ]:
# Escriba aquí su solución.
